# Rasch Workflow (ltm-style)

This notebook mirrors the **R `ltm`** workflow using the LSAT dataset.

In R, a typical ltm workflow looks like:

```r
library(ltm)
data(LSAT)
mod_rasch <- rasch(LSAT, constraint = cbind(ncol(LSAT) + 1, 1))
coef(mod_rasch)
factor.scores(mod_rasch, method = "EAP")
```

Below we do the same steps in Python with this IRT module.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from irt import fit

## 1) Load the LSAT data (same data used by ltm)

The file below was generated from the R `ltm` package and stored in `tests/ltm_reference/`.

In [2]:
lsat = pd.read_csv("../tests/ltm_reference/lsat_data.csv")
lsat.shape

(1000, 5)

## 2) Fit Rasch with ltm-like settings

`ltm` uses fewer quadrature points by default (often 21). We can match that with `technical={"quadpts": 21}` for a closer comparison.

In [3]:
rasch_result = fit(
    lsat,
    model="rasch",
    estimator="mml_em",
    technical={"quadpts": 21}
)
rasch_result

FitResult(model='rasch', estimator='mml_em', n_items=5, n_persons=1000, n_iter=10, converged, loglik=-2473.07)

## 3) Compare item difficulties to ltm reference

`ltm` and this module both use the same IRT parameterization. We still **center** difficulties before comparing because each package can choose a slightly different origin.

In [4]:
ref_params = pd.read_csv("../tests/ltm_reference/lsat_rasch_fixed_params.csv")

b_py = rasch_result.params["b"]
b_ref = ref_params["b"].values

b_py_centered = b_py - b_py.mean()
b_ref_centered = b_ref - b_ref.mean()

corr = np.corrcoef(b_py_centered, b_ref_centered)[0, 1]
mad = np.abs(b_py_centered - b_ref_centered).mean()

corr, mad

(np.float64(0.9999999999849825), np.float64(2.1803918985563443e-05))

## 4) Item and person summaries

These are the same kinds of outputs you would typically examine in `ltm`.

In [5]:
rasch_result.item_report(), rasch_result.person_report().head()

(     item    a         b  n_obs
 0  Item 1  1.0 -2.871865   1000
 1  Item 2  1.0 -1.062981   1000
 2  Item 3  1.0 -0.257576   1000
 3  Item 4  1.0 -1.388004   1000
 4  Item 5  1.0 -2.218701   1000,
   person  n_items  sum_score     theta        se
 0      0        5        0.0 -2.033624  0.711446
 1      1        5        0.0 -2.033624  0.711446
 2      2        5        0.0 -2.033624  0.711446
 3      3        5        1.0 -1.527829  0.711260
 4      4        5        1.0 -1.527829  0.711260)

## 5) EAP and EB (MAP) scores

In `ltm`, **EB** is the same idea as **MAP**. We'll compute both and compare to the reference outputs.

In [6]:
scores_eap = rasch_result.score(method="eap")
scores_map = rasch_result.score(method="map")

ltm_eap = pd.read_csv("../tests/ltm_reference/lsat_rasch_fixed_eap.csv")
ltm_eb = pd.read_csv("../tests/ltm_reference/lsat_rasch_fixed_eb.csv")

eap_corr = np.corrcoef(scores_eap.theta, ltm_eap["theta"].values)[0, 1]
map_corr = np.corrcoef(scores_map.theta, ltm_eb["theta"].values)[0, 1]

eap_corr, map_corr

(np.float64(0.9999999407782664), np.float64(0.9906286424902232))

## What you learned

- How to reproduce the **ltm Rasch workflow** in Python
- How to compare item difficulties and ability scores to R reference outputs
- How to use `technical` settings to align quadrature choices

Next: `02_mirt_style_2pl.ipynb` for the mirt-style Rasch and 2PL workflows.